In [2]:
import pandas as pd

CN_2014_2024 = pd.read_csv('dataset/CN_2014-2024.csv', sep = ',')
CN_202501_202508 = pd.read_csv('dataset/CN_202501-202508.csv', sep = ',')

cn_pf  = pd.concat([CN_2014_2024, CN_202501_202508], axis = 0)
cn_pf =   cn_pf[cn_pf['name']=='Heilongjiang' ]

In [3]:
# 数据预处理
cn_pf['date'] = pd.to_datetime(cn_pf['date'], errors='coerce')
cn_pf['year'] = cn_pf['date'].dt.year
cn_pf['day_of_year'] = cn_pf['date'].dt.dayofyear
cn_pf['day_lab'] = cn_pf['date'].dt.strftime('%m-%d')

cn_pf = cn_pf[ cn_pf['day_lab'] != '02-29']

# 年份
years = cn_pf['year'].unique()
this_year = years.max()
years = years[years < this_year]

# 历史数据
df_history = cn_pf[cn_pf['year'] != this_year]

extremes = df_history.groupby('day_lab')['soil_moisture_7_to_28cm_mean'].agg(['min', 'max', 'mean']).reset_index()

# x =extremes['day_of_year'].astype(str)



In [4]:
from pyecharts import options as opts
from pyecharts.charts import Line
from pyecharts.commons.utils import JsCode


# 设置 x 轴
line = Line(init_opts=opts.InitOpts(width="800px", height="480px"))
# 配置项
line.set_global_opts(
    legend_opts=opts.LegendOpts(is_show=True),
    tooltip_opts=opts.TooltipOpts(trigger="axis"),
    toolbox_opts=opts.ToolboxOpts(is_show=True,
                                  feature=opts.ToolBoxFeatureOpts(
                                      save_as_image=opts.ToolBoxFeatureSaveAsImageOpts(background_color='#FFFFFF')
                                  )),
    datazoom_opts=opts.DataZoomOpts(is_show=True, range_start=0, range_end=100),
)

line.add_xaxis(xaxis_data=extremes['day_lab'].tolist())

line.add_yaxis(
    series_name="Min",
    y_axis=extremes['min'].round(4).tolist(),
    # is_smooth=True,
    is_symbol_show=False,
    linestyle_opts=opts.LineStyleOpts(opacity=0),
    areastyle_opts=opts.AreaStyleOpts(opacity=0, color="rgba(0, 0, 0, 0)"),
    stack="堆叠",
    label_opts=opts.LabelOpts(is_show=False)
)

a = (extremes['max']-extremes['min']).round(4).tolist()

line.add_yaxis(
    series_name="max",
    y_axis=a,
    # is_smooth=True,
    is_symbol_show=False,
    linestyle_opts=opts.LineStyleOpts(opacity=0),
    areastyle_opts=opts.AreaStyleOpts(opacity=0.4, color="skyblue"),
    stack="堆叠",
    label_opts=opts.LabelOpts(is_show=False)
)

line.add_yaxis(
    series_name="均值",
    y_axis=extremes['mean'].round(4).tolist(),
    # is_smooth=True,
    is_symbol_show=False,
    linestyle_opts=opts.LineStyleOpts(width=1.2, type_="dashed", color="black"),
    areastyle_opts=opts.AreaStyleOpts(opacity=0),
    # stack="堆叠",
    label_opts=opts.LabelOpts(is_show=False)
)

for year in years:
    year_data = df_history[(df_history['year'] == year) & (df_history['year'] >= 2023)]
    if not year_data.empty:
        line.add_yaxis(
            series_name=str(year),
            y_axis=year_data['soil_moisture_7_to_28cm_mean'].round(4).tolist(),
            is_smooth=True,
            linestyle_opts=opts.LineStyleOpts(width=1.2),
            symbol="none",
            is_symbol_show=False,

        )


line.add_yaxis(
    series_name=str(this_year),
    y_axis= cn_pf[ cn_pf['year'] == this_year ]['soil_moisture_7_to_28cm_mean'].round(4).tolist(),
    is_smooth=True,
    linestyle_opts=opts.LineStyleOpts(width=1.5, color="red"),
    symbol="none",
)

line.render('test.html')

line.render_notebook()


In [35]:
# line.render('test.html')

'/Users/xianr/DataspellProjects/weather/test.html'

In [38]:
import pyecharts.options as opts
from pyecharts.charts import Line

"""
Gallery 使用 pyecharts 1.1.0
参考地址: https://echarts.apache.org/examples/editor.html?c=area-stack

目前无法实现的功能:

暂无
"""




x_data = extremes['day_of_year'].tolist()

(
    Line()
    .add_xaxis(xaxis_data=x_data)
    .add_yaxis(
        series_name="最小值",
        stack="总量",
        y_axis=extremes['min'].tolist(),
        areastyle_opts=opts.AreaStyleOpts(opacity=0.5),
        label_opts=opts.LabelOpts(is_show=False),
    )
    .add_yaxis(
        series_name="最大值",
        stack="总量",
        y_axis=(extremes['max']-extremes['min']).tolist(),
        areastyle_opts=opts.AreaStyleOpts(opacity=0.5),
        label_opts=opts.LabelOpts(is_show=False),
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="堆叠区域图"),
        tooltip_opts=opts.TooltipOpts(trigger="axis", axis_pointer_type="cross"),
        yaxis_opts=opts.AxisOpts(
            type_="value",
            axistick_opts=opts.AxisTickOpts(is_show=True),
            splitline_opts=opts.SplitLineOpts(is_show=True),
        ),
        xaxis_opts=opts.AxisOpts(type_="category", boundary_gap=False),
    )
    .render("stacked_area_chart.html")
)


'/Users/xianr/DataspellProjects/weather/stacked_area_chart.html'